# 02 — Train Models

Trains L2-regularized logistic regression and LightGBM for in-hospital mortality prediction, using **TRAIN** for fitting and **VALIDATION ONLY** for hyperparameter selection and early stopping. **The test set is never loaded in this notebook.**

Class imbalance is handled via `class_weight="balanced"` (logistic regression) and a dynamically computed `scale_pos_weight` (LightGBM, from TRAIN class counts only).

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path

import joblib
import pandas as pd

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.preprocess import (
    load_cohort_features, drop_duplicate_rows, patient_level_split, check_patient_overlap,
    drop_identifier_columns, identify_feature_types, IDENTIFIER_COLUMNS,
)
from src.train_paper_models import (
    train_logistic_regression, train_lightgbm, build_hyperparameters_table, build_long_format_predictions,
)

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("02_train_models started (seed=%d)", seed)

ID_COL = config["preprocessing"]["id_column"]
TARGET_COL = config["preprocessing"]["target_column"]
MODELS_DIR = Path(config["training"]["models_output_dir"])
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Reproduce the split, load the frozen preprocessor

Reproduces the identical patient-level split from `01_cohort_audit.ipynb` (same data, same seed) and loads the preprocessor fit there — it is NOT refit here.

In [ ]:
raw_df = load_cohort_features(config=config)
dedup_df = drop_duplicate_rows(raw_df)

train_df, val_df, test_df = patient_level_split(
    dedup_df, id_col=ID_COL, target_col=TARGET_COL,
    train_size=config["preprocessing"]["train_size"],
    val_size=config["preprocessing"]["val_size"],
    test_size=config["preprocessing"]["test_size"],
    seed=seed,
)
check_patient_overlap(train_df, val_df, test_df, id_col=ID_COL)
del test_df  # not used in this notebook

preprocessor = joblib.load(config["preprocessing"]["preprocessor_output_path"])

train_features_df = drop_identifier_columns(train_df, IDENTIFIER_COLUMNS)
val_features_df = drop_identifier_columns(val_df, IDENTIFIER_COLUMNS)

X_train = preprocessor.transform(train_features_df)
X_val = preprocessor.transform(val_features_df)
y_train = train_features_df[TARGET_COL].reset_index(drop=True)
y_val = val_features_df[TARGET_COL].reset_index(drop=True)

print(f"X_train: {X_train.shape} (positive rate {y_train.mean():.1%})")
print(f"X_val:   {X_val.shape} (positive rate {y_val.mean():.1%})")

## 2. Train logistic regression (grid search over C, validation AUROC)

In [ ]:
logreg_result = train_logistic_regression(X_train, y_train, X_val, y_val, config, logger)
logreg_result["search_log"]

## 3. Train LightGBM (grid search over learning_rate, early stopping on validation)

In [ ]:
lightgbm_result = train_lightgbm(X_train, y_train, X_val, y_val, config, logger)
lightgbm_result["search_log"]

## 4. Save trained models

In [ ]:
results = [logreg_result, lightgbm_result]

model_filenames = {"logistic_regression": "model_logreg.joblib", "lightgbm": "model_lgbm.joblib"}
for result in results:
    out_path = MODELS_DIR / model_filenames[result["model_name"]]
    joblib.dump(result["model"], out_path)
    logger.info("[%s] model saved to %s", result["model_name"], out_path)
    print(f"Saved {result['model_name']} -> {out_path}")

## 5. Save validation predictions (long format)

`outputs/predictions/val_predictions.parquet` — columns `stay_id`, `model_name`, `predicted_prob`, `true_label`.

In [ ]:
val_predictions_df = build_long_format_predictions(results, val_df, y_val, prob_key="val_prob")

val_predictions_path = Path(config["training"]["val_predictions_path"])
val_predictions_path.parent.mkdir(parents=True, exist_ok=True)
val_predictions_df.to_parquet(val_predictions_path, index=False)
logger.info("Validation predictions saved to %s (%d rows)", val_predictions_path, len(val_predictions_df))

print(f"Saved {val_predictions_path} ({len(val_predictions_df)} rows)")
val_predictions_df.head()

## 6. Table 2: hyperparameters

In [ ]:
hyperparams_df = build_hyperparameters_table(results, seed)
hyperparams_path = Path(config["training"]["hyperparameters_table_path"])
hyperparams_path.parent.mkdir(parents=True, exist_ok=True)
hyperparams_df.to_csv(hyperparams_path, index=False)
logger.info("Hyperparameters table saved to %s", hyperparams_path)

hyperparams_df

## 7. Run metadata

In [ ]:
log_run_metadata(
    seed=seed,
    extra={
        "notebook": "02_train_models",
        "n_train": len(train_df), "n_val": len(val_df),
        "models": {r["model_name"]: {"val_auroc": round(r["val_auroc"], 4), "hyperparameters": r["hyperparameters"]} for r in results},
        "test_set_touched": False,
    },
)
logger.info("02_train_models finished")
for r in results:
    print(f"{r['model_name']}: val AUROC = {r['val_auroc']:.4f}")